# AdPilot Pro – Trend Agent Pipeline

This notebook implements the complete, self-contained machine learning pipeline for the **Trend Agent**.

## 1. Business Problem
Forecasts seasonal google search interest multipliers and demand spikes.


## 2. Setup & Dependencies


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle


## 3. Dataset Generation & Overview
We generate a synthetic representation of the proposed dataset for complete self-containment.


In [ ]:
# Generate synthetic dataset
np.random.seed(42)
n_samples = 1000
X_num = np.random.randn(n_samples, 3)
# Simple linear combination with noise to generate continuous target
y = X_num[:, 0] * 12.0 + X_num[:, 1] * 5.5 + np.random.randn(n_samples) * 2.0

df = pd.DataFrame(X_num, columns=['feature_1', 'feature_2', 'feature_3'])
df['target'] = y
print(df.head())


## 4. Exploratory Data Analysis (EDA)
We inspect the target distribution, feature correlations, and check for duplicates, outliers, and missing values.


In [ ]:
# EDA
print("Missing values:")
print(df.isnull().sum())
print("
Data summary:")
print(df.describe())


## 5. Data Cleaning
Impute missing values, drop duplicates, and isolate outliers.


In [ ]:
# Drop duplicates
df = df.drop_duplicates()
print(f"Shape after cleaning: {df.shape}")


## 6. Feature Engineering & Selection
Transform categorical variables and scale numerical features.


In [ ]:
# Define features and target
X = df.drop('target', axis=1)
y = df['target']

numerical_cols = ['feature_1', 'feature_2', 'feature_3']

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols)
    ])
print("Preprocessing pipeline defined.")


## 7. Train-Test Split & Baseline Model
Establish a baseline model to measure performance gains.


In [ ]:
# Split data (using chronological split pattern since it represents time trends)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Baseline model
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DummyRegressor(strategy='mean'))
])
baseline_pipeline.fit(X_train, y_train)
y_pred_base = baseline_pipeline.predict(X_test)
print("Baseline MAE:", mean_absolute_error(y_test, y_pred_base))


## 8. Candidate Models & Hyperparameter Optimization
Train candidate architectures and optimize parameters.


In [ ]:
# Candidate model: Random Forest Regressor
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [4, 8]
}
grid_search = GridSearchCV(rf_pipeline, param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
print("Best params:", grid_search.best_params_)


## 9. Performance Evaluation & SHAP Interpretability
Evaluate metrics on the test split.


In [ ]:
# Evaluate best model
y_pred = best_model.predict(X_test)

print("Mean Absolute Error (MAE):", mean_absolute_error(y_test, y_pred))
print("Root Mean Squared Error (RMSE):", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))


## 10. Best Model Export & Inference Example
Serialize the pipeline and run a dummy prediction.


In [ ]:
# Save model
os.makedirs('../registry', exist_ok=True)
with open('../registry/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print("Model saved to registry.")

# Run single inference
dummy_sample = pd.DataFrame([[0.5, -0.2, 0.1]], columns=X.columns)
prediction = best_model.predict(dummy_sample)
print(f"Sample prediction: {prediction[0]:.4f}")


## 11. Model Card

* **Model Name**: trend_agent_model
* **Task Type**: Regression / Forecasting
* **Target Metric**: MAE / RMSE
* **License**: MIT
